# Migration: Dept_pkg
**Conversion Date:** 2024-05-22
**Source File:** TARGET_ODI_SQL.sql

In [ ]:
dbutils.widgets.text("V_Dept", "")
dbutils.widgets.text("ODI_SESS_NO", "")

## ETL Parameters

In [ ]:
%sql
-- SCEN_TASK_NO {1}: Get ETL last extract time
CREATE OR REPLACE TEMPORARY VIEW v_dept_param AS
SELECT COALESCE(
    (SELECT date_format(last_run_dt, 'dd-MM-yy') 
     FROM workspace.odi_src.log_table1 
     WHERE pkg_name = 'Dept_pkg'),
    date_format(current_timestamp(), 'dd-MM-yy')
) AS v_dept_val;

## Staging Table: C$_0FILTER

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Drop staging table
DROP TABLE IF EXISTS workspace.odi_src.c_0filter;

In [ ]:
%sql
-- SCEN_TASK_NO {40}: Create staging table
CREATE TABLE workspace.odi_src.c_0filter
(
	DEPARTMENT_ID	BIGINT,
	DEPARTMENT_NAME	STRING,
	MANAGER_ID	BIGINT,
	LOCATION_ID	BIGINT,
	LAST_UPD_DT	TIMESTAMP
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {50}: Insert into staging
INSERT INTO workspace.odi_src.c_0filter
(
	DEPARTMENT_ID,
	DEPARTMENT_NAME,
	MANAGER_ID,
	LOCATION_ID,
	LAST_UPD_DT
)
SELECT 	
	SRC_DEPARTMENTS_1.DEPARTMENT_ID	AS DEPARTMENT_ID,
	SRC_DEPARTMENTS_1.DEPARTMENT_NAME	AS DEPARTMENT_NAME,
	SRC_DEPARTMENTS_1.MANAGER_ID	AS MANAGER_ID,
	SRC_DEPARTMENTS_1.LOCATION_ID	AS LOCATION_ID,
	SRC_DEPARTMENTS_1.LAST_UPD_DT	AS LAST_UPD_DT
FROM	workspace.odi_src.src_departments AS SRC_DEPARTMENTS_1
WHERE	(1=1)
 AND (SRC_DEPARTMENTS_1.LAST_UPD_DT >= to_date('${V_Dept}', 'dd-MM-yy'));

In [ ]:
%sql
-- SCEN_TASK_NO {60}: Gather Statistics / Optimize
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.odi_src.c_0filter;

## Flow Table: I$_TRG_DEPARTMENTS

In [ ]:
%sql
-- SCEN_TASK_NO {80}: Drop flow table
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;

In [ ]:
%sql
-- SCEN_TASK_NO {90}: Create flow table
CREATE TABLE workspace.odi_trg.i_trg_departments_flow
(
	DEPARTMENT_ID	BIGINT,
	DEPARTMENT_NAME	STRING,
	MANAGER_ID	BIGINT,
	LOCATION_ID	BIGINT,
	LAST_UPD_DT	TIMESTAMP,
	IND_UPDATE		STRING
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO {100}: Insert into flow table
INSERT INTO	workspace.odi_trg.i_trg_departments_flow
(
	DEPARTMENT_ID,
	DEPARTMENT_NAME,
	MANAGER_ID,
	LOCATION_ID,
	LAST_UPD_DT,
	IND_UPDATE
)
SELECT 	
	FILTER_A.DEPARTMENT_ID,
	FILTER_A.DEPARTMENT_NAME,
	FILTER_A.MANAGER_ID,
	FILTER_A.LOCATION_ID,
	FILTER_A.LAST_UPD_DT,
	'I' AS IND_UPDATE
FROM	workspace.odi_trg.c_0filter AS FILTER_A
WHERE	(1=1);

## Cleanup

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.odi_src.c_0filter;
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;